# Бейзлайн: логистическая регрессия с полным пайплайном
Цель: быстро проверить общий сигнал по таргету `Рез_экзамена` в полном пайплайне обработки.


In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "README.md").exists()
    and (path / "Data_making").is_dir()
    and (path / "Models").is_dir()
)
DATA_PATH = (
    PROJECT_ROOT
    / "Data_making"
    / "synthetic_education_dushanbe_WITH_ROUNDED.csv"
)
METRICS_DIR = PROJECT_ROOT / "Models" / "Compare models"
METRICS_DIR.mkdir(parents=True, exist_ok=True)

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix


## Данные
Коротко: читаем таблицу и проверяем, что загрузка прошла корректно.


In [ ]:
# Данные
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
# Общий набор признаков для честного сравнения моделей
TARGET_COL = "Рез_экзамена"
ID_COL = "ID_ученика"
LEAKAGE_RISK_COLS = ["Средний_балл"]
FEATURE_COLUMNS = [
    "Класс",
    "Район",
    "Часы_самоподготовки_в_неделю",
    "Посещаемость_%",
    "Уверенность_в_себе",
    "Уровень_стресса_перед_экзаменом",
    "Пропущенные_дни",
    "Тип_школы",
    "Индекс_качества_школы",
    "Стабильность_преподавателей",
    "Доступ_к_ресурсам",
    "Образовательная_среда",
]
y = df[TARGET_COL]
X = df[FEATURE_COLUMNS].copy()

print('Исключены потенциальные leakage-признаки:', LEAKAGE_RISK_COLS)
print('Форма X после очистки:', X.shape)

# Проверка баланса классов
y.value_counts(normalize=True)


## Таргет и признаки
Цель — `Рез_экзамена`. `ID_ученика` исключаем, чтобы не было утечки.


In [ ]:
# Разделение
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# Определяем признаки
categorical_features = X.select_dtypes(include=['object']).columns
numeric_features = X.select_dtypes(exclude=['object']).columns

## Препроцессинг + модель
Числовые признаки: медиана + стандартизация. Категориальные: мода + OHE.


In [ ]:
# Пайплайн
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')) ,
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')) ,
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

model = LogisticRegression(max_iter=500, random_state=42, class_weight='balanced')

clf = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', model)
])

In [ ]:
# Обучение
clf.fit(X_train, y_train)

## Оценка
Смотрим базовые метрики и матрицу ошибок на отложенной выборке.

In [ ]:
# Оценка
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

print('Accuracy:', accuracy_score(y_test, y_pred))
print('ROC AUC:', roc_auc_score(y_test, y_proba))
print('\nClassification report:\n', classification_report(y_test, y_pred))
print('Confusion matrix:\n', confusion_matrix(y_test, y_pred))

## Кросс-валидация
Проверяем устойчивость метрики на фолдах.

In [ ]:
# Кросс-валидация
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(clf, X, y, cv=cv, scoring='roc_auc')
print('CV ROC AUC: %.3f ± %.3f' % (scores.mean(), scores.std()))

## Визуализация метрик (Plotly)
Покажем основные метрики, ROC-кривую и матрицу ошибок в понятном виде.


In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import precision_score, recall_score, f1_score, roc_curve

In [ ]:
# Основные метрики
metrics = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'ROC AUC': roc_auc_score(y_test, y_proba),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1': f1_score(y_test, y_pred)
}
metrics_df = pd.DataFrame({'metric': list(metrics.keys()), 'value': list(metrics.values())})
fig_m = px.bar(metrics_df, x='metric', y='value', text='value', title='Основные метрики модели')
fig_m.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig_m.update_layout(yaxis=dict(range=[0, 1]))
fig_m.show()

In [ ]:
# ROC-кривая
fpr, tpr, _ = roc_curve(y_test, y_proba)
fig_roc = go.Figure()
fig_roc.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name='ROC'))
fig_roc.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Random', line=dict(dash='dash')))
fig_roc.update_layout(title='ROC-кривая', xaxis_title='FPR', yaxis_title='TPR')
fig_roc.show()

In [ ]:
# Матрица ошибок
cm = confusion_matrix(y_test, y_pred)
fig_cm = px.imshow(cm, text_auto=True, aspect='auto',
                   labels=dict(x='Predicted', y='Actual', color='Count'),
                   title='Матрица ошибок')
fig_cm.show()

## SHAP (интерпретация модели)
Покажем важность признаков и вклад в предсказание.


In [ ]:
import shap

preprocess = clf.named_steps['preprocess']
model = clf.named_steps['model']

X_train_t = preprocess.transform(X_train)
X_test_t = preprocess.transform(X_test)

feature_names = preprocess.get_feature_names_out()

explainer = shap.LinearExplainer(model, X_train_t, feature_perturbation='interventional')
shap_values = explainer(X_test_t)

shap.summary_plot(shap_values, features=X_test_t, feature_names=feature_names, show=True)

## Сохранение метрик для сравнения


In [ ]:
import json
# Метрики логистической регрессии (из уже вычисленных значений)
baseline_metrics = {
    'model': 'LogReg',
    'AUC': float(roc_auc_score(y_test, y_proba)),
    'Accuracy': float(accuracy_score(y_test, y_pred)),
    'Precision': float(precision_score(y_test, y_pred)),
    'Recall': float(recall_score(y_test, y_pred)),
    'F1': float(f1_score(y_test, y_pred)),
    'CV_AUC_mean': float(scores.mean()),
    'CV_AUC_std': float(scores.std()),
    'feature_scenario': 'common_no_avg_grade',
    'feature_columns': FEATURE_COLUMNS,
    'excluded_leakage_risk_cols': LEAKAGE_RISK_COLS
}

out_path = METRICS_DIR / "logreg_metrics.json"
out_path.write_text(json.dumps(baseline_metrics, ensure_ascii=False, indent=2), encoding="utf-8")
print('saved', out_path)
